# Card 2024 · speech T15

## Prepare the data

Run the commands below from the repository root with `bci-repro` active. See the [reader guide](../../docs/20261124_repository_migration/reader_workflow.md) for environment setup.

### 1. Download

Download `t15_copyTask_neuralData.zip` from [Dryad](https://doi.org/10.5061/dryad.dncjsxm85). Create `studies/20240815_card/data/raw/dryad_dncjsxm85/` and save the download as `studies/20240815_card/data/raw/dryad_dncjsxm85/dryad_dncjsxm85__t15_copyTask_neuralData.zip`.

### 2. Extract

Extract the archive into `studies/20240815_card/data/extracted/dryad_dncjsxm85/`.

```text
python -m zipfile -e "studies/20240815_card/data/raw/dryad_dncjsxm85/dryad_dncjsxm85__t15_copyTask_neuralData.zip" "studies/20240815_card/data/extracted/dryad_dncjsxm85"
```

After extraction, check for `studies/20240815_card/data/extracted/dryad_dncjsxm85/hdf5_data_final/`.

### 3. Prepare the model input

The trainer reads the released HDF5 directly from `studies/20240815_card/data/extracted/dryad_dncjsxm85/hdf5_data_final/`. This directory is the HDF5 input for training and inference.

Set up the upstream trainer. For inference or logit export, copy `studies/20240815_card/open_source_code_for_reference/nejm-brain-to-text/data/t15_copyTaskData_description.csv` into `studies/20240815_card/data/extracted/dryad_dncjsxm85/`.

```text
python shared/tools/setup_upstream.py --study 20240815_card
```

### 4. Export the common session file

Write one recording day to the session format shared by every study in this repository. The file stores each labeled trial with the features as released and records the publisher file and the position the trial came from. The trials of the day's withheld-label `data_test.hdf5` file are counted in the root attributes and not exported. This example exports August 11, 2023.

```text
python studies/20240815_card/code/export_sessions.py --session t15.2023.08.11
```

**Output** `studies/20240815_card/data/processed/t15/dryad_dncjsxm85_2023-08-11.h5`. Add `--list` to see every session and whether it is exported, or use `--all` to export them all.

Check the file structure and trace every trial back to its publisher file with one command.

```text
python shared/tools/validate_session.py studies/20240815_card/data/processed/t15/dryad_dncjsxm85_2023-08-11.h5 --source trials
```

**Output** A `PASS` line for the file, ending with `source trace: 288/288 trials re-read and identical`.

In [1]:
import sys
from pathlib import Path

root = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "figures/reproduction_benchmark.yaml").is_file())
sys.path.insert(0, str(root / "shared"))
from session_format import read_session

session_file = root / "studies/20240815_card/data/processed/t15/dryad_dncjsxm85_2023-08-11.h5"
if session_file.is_file():
    with read_session(session_file) as session:
        print(f"{session.attrs['session']}: {len(session)} trials")
        trial = session.trials[0]
        print(trial.trial_id, "·", trial.target("sentence"))
        array = trial.neural("input_features")
        print(f"  neural/input_features: {array.shape} {array.dtype}")
        labels = session.label_sets["phonemes"]
        print("  phonemes:", " ".join(labels[i].strip() for i in trial.target("phonemes")[:10]))
        print(f"  corpus {trial.attrs['corpus']}, block {trial.attrs['source_block']}, "
              f"trial {trial.attrs['trial_num']}")
else:
    print("Session file not found. Run the export command above to create it.")

t15.2023.08.11: 288 trials
t15.2023.08.11/train/0000 · Bring it closer.
  neural/input_features: (321, 512) float32
  phonemes: B R IH NG | IH T | K L
  corpus 50-Word, block 2, trial 0


## Read the archived results

Figure 3a uses the validation trials and day-index-corrected decode from the P4 GRU. Figure 2f uses the BiGRU comparison. The rapid-calibration results are a separate archive.

This walkthrough uses the checked-in archive and takes no data download.

In [2]:
import csv
import json
import sys
from pathlib import Path

root = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "figures/reproduction_benchmark.yaml").is_file())
sys.path.insert(0, str(root / "figures/code"))
from figure_paths import PUBLISHED, load_benchmark

benchmark = load_benchmark()

def read_rows(path):
    """Read the archived CSV rows in their original order."""
    with (root / path).open(encoding="utf-8-sig", newline="") as stream:
        return list(csv.DictReader(stream))

## From a trial or day to the reported result

In [3]:
arm = "Card 2024 (T15)"
from build_session_per_vs_wer import build, read_tsv, word_ed

source = benchmark["figure3_per_sources"][arm]
rows = read_rows(source["greedy"])
decoded = read_tsv(root / source["decode"])
trial = rows[0]
reference, hypothesis = decoded[0]
print(f"{trial['session']} · {reference}")
print(f"Word hypothesis: {hypothesis}")
edits, words = word_ed(reference, hypothesis)
print(f"Trial word edits: {edits}/{words}")
result = build(arm, root / source["greedy"], root / source["decode"])
day = result["sessions"][0]
print(f"{day['session']}: {day['n']} trials, PER {day['per']:.2f}%, WER {day['wer']:.2f}%")
print(f"{len(result['sessions'])} sessions, {len(rows)} evaluated trials; pooled {result['pooled']}")

t15.2023.08.13 · You can see the code at this point as well.
Word hypothesis: you can see the code at this point as well
Trial word edits: 0/10
Card 2024 (T15)   : 41 sessions | pooled PER 10.31% | WER  7.02%
t15.2023.08.13: 35 trials, PER 6.44%, WER 3.48%
41 sessions, 1426 evaluated trials; pooled {'per': 10.31, 'wer': 7.02}


## Reproduce and score

[Common reader guide](../../docs/20261124_repository_migration/reader_workflow.md) · [Study evaluation protocol](docs/20261124_evaluation_protocol/README.md)

### Train or use released weights

The trainer reads the released `hdf5_data_final/` directly. Run in PowerShell at the repository root.

```powershell
$study = "studies/20240815_card"
$date = Get-Date -Format yyyyMMdd
$run = "$study/results/${date}_dryad_dncjsxm85__hdf5__ctc_greedy__reader/runtime"
python "$study/code/train.py" 1000 "$run" 1000
python "$study/code/infer.py" "$run"
```

For the released weights, download `t15_pretrained_rnn_baseline.zip` from the same Dryad release. Save it under `data/raw/dryad_dncjsxm85/` as `dryad_dncjsxm85__t15_pretrained_rnn_baseline.zip`.

```powershell
python -m zipfile -e "$study/data/raw/dryad_dncjsxm85/dryad_dncjsxm85__t15_pretrained_rnn_baseline.zip" "$study/data/checkpoints/dryad_dncjsxm85"
```

Point the inference command at `data/checkpoints/dryad_dncjsxm85/t15_pretrained_rnn_baseline/` within the study. The inference output is `t15_logits_fixed.hdf5` in the directory passed to `infer.py`.

### Calculate metrics

Read the validation PER printed during training. For overall WER from fresh logits, follow [speech word decoding in WSL](../../docs/20261124_repository_migration/reader_workflow.md#speech-word-error-rate-in-wsl) with the Card row. The shared scorer prints the total word edits, reference words, and WER. The [evaluation protocol](docs/20261124_evaluation_protocol/README.md) identifies the archived day-index-corrected P4 and BiGRU results.
